In [ ]:
#Import everything

import os
import torch
import random
import numpy as np
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using:", device)

Using: cuda


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Adjust the path to your dataset within the mounted Google Drive
DATA_DIR =  "/content/drive/MyDrive/Coconut Tree Disease Dataset" # Assuming the folder is directly in MyDrive
#print(os.listdir('/content/drive/MyDrive'))
print(DATA_DIR)

Mounted at /content/drive
/content/drive/MyDrive/Coconut Tree Disease Dataset


In [ ]:
# Standardize dataset
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],
                         [0.229, 0.224, 0.225])
])
print("Happened")

Happened


In [ ]:
# Split the data
full_dataset = datasets.ImageFolder(DATA_DIR, transform=train_transform)

class_names = full_dataset.classes
num_classes = len(class_names)

print("Classes:", class_names)

# 80/20 split
train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = random_split(full_dataset, [train_size, val_size])

# Apply val transforms separately
val_dataset.dataset.transform = val_transform

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

Classes: ['Bud Root Dropping', 'Bud Rot', 'Gray Leaf Spot', 'Leaf Rot', 'Stem Bleeding']


In [ ]:
from torchvision import models
import torch.nn as nn
import torch.optim as optim
# Make the model using torchvision.models.resnet18
model = models.resnet18(pretrained=True) # Use a pre-trained PyTorch ResNet18

# Freeze base layers
for param in model.parameters():
    param.requires_grad = False

# Replace final layer
# The last layer in torchvision's resnet18 is named 'fc'
num_ftrs = model.fc.in_features
model.fc = nn.Linear(num_ftrs, num_classes)

model = model.to(device)
criterion = nn.CrossEntropyLoss()
# Only optimize parameters of the new final layer
optimizer = optim.Adam(model.fc.parameters(), lr=0.001)
print("Happened")

/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 242MB/s]


Happened


In [ ]:
# train_model method
def train_model(model, train_loader, val_loader, epochs=10):
    best_acc = 0.0

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        # TRAIN
        model.train()
        running_loss = 0.0
        correct = 0

        for images, labels in tqdm(train_loader):
            images, labels = images.to(device), labels.to(device)

            optimizer.zero_grad()

            outputs = model(images)
            loss = criterion(outputs, labels)

            loss.backward()
            optimizer.step()

            running_loss += loss.item()
            _, preds = torch.max(outputs, 1)
            correct += torch.sum(preds == labels)

        train_acc = correct.double() / len(train_loader.dataset)

        # VALIDATION
        model.eval()
        val_loss = 0.0
        val_correct = 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)

                outputs = model(images)
                loss = criterion(outputs, labels)

                val_loss += loss.item()
                _, preds = torch.max(outputs, 1)
                val_correct += torch.sum(preds == labels)

        val_acc = val_correct.double() / len(val_loader.dataset)

        print(f"Train Loss: {running_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")

        if val_acc > best_acc:
            best_acc = val_acc
            torch.save(model.state_dict(), "best_model.pth")
            print("✅ Saved best model")

    print("Best Validation Accuracy:", best_acc)

In [ ]:
# predict_image method
from PIL import Image

def predict_image(image_path):
    model.eval()

    image = Image.open(image_path).convert("RGB")
    image = val_transform(image).unsqueeze(0).to(device)

    with torch.no_grad():
        outputs = model(image)
        _, pred = torch.max(outputs, 1)

    return class_names[pred.item()]

# Test
#print(predict_image("/kaggle/input/test-image.jpg"))

In [ ]:
# Trains the model
print(next(model.parameters()).device)
train_model(model, train_loader, val_loader, epochs=10)
print("Happened")

cuda:0

Epoch 1/10


  7%|▋         | 10/146 [03:39<42:03, 18.56s/it]